# Phase 0+1 — Environment Setup, Dataset Download & Shared Data Split

**Project:** RealWaste Waste Classification  
**Course:** Introduction to Machine Learning (IML)

This notebook:
1. Detects the runtime environment (Colab / local)
2. Installs all required dependencies
3. Mounts Google Drive (on Colab) for persistent storage
4. Downloads the RealWaste dataset from UCI
5. Checks for duplicate images
6. Creates a single stratified train/val/test split used by ALL notebooks
7. Saves the split index files so every model uses identical data

> ⚠️ **Run this notebook ONCE before any other notebook.**

## 0.1 — Detect Environment

In [ ]:
import sys

IS_COLAB  = 'google.colab' in sys.modules
IS_KAGGLE = 'kaggle_secrets' in sys.modules
IS_LOCAL  = not IS_COLAB and not IS_KAGGLE

env_name = 'Google Colab' if IS_COLAB else ('Kaggle' if IS_KAGGLE else 'Local')
print(f'Running on: {env_name}')
print(f'Python version: {sys.version}')

## 0.2 — Install Dependencies

In [ ]:
# Install all required packages
# torch + torchvision are pre-installed on Colab; still safe to run
%pip install -q torch torchvision
%pip install -q ultralytics
%pip install -q scikit-learn matplotlib seaborn pillow tqdm pandas numpy

print('✅ All dependencies installed.')

## 0.3 — Mount Google Drive (Colab only)

Everything will be saved under `My Drive/IML_RealWaste/` so it persists across Colab sessions.

In [ ]:
from pathlib import Path

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # All project files will live here on Drive
    PROJECT_ROOT = Path('/content/drive/MyDrive/IML_RealWaste')
else:
    # Local: use the repo root (one level up from notebooks/)
    PROJECT_ROOT = Path('.').resolve().parent

# Subdirectories
DATA_DIR    = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
METRICS_DIR = RESULTS_DIR  / 'metrics'
PLOTS_DIR   = RESULTS_DIR  / 'plots'
MODELS_DIR  = RESULTS_DIR  / 'models'

for d in [DATA_DIR, METRICS_DIR, PLOTS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Results dir  : {RESULTS_DIR}')

## 0.4 — Download RealWaste Dataset

Dataset: [RealWaste — UCI ML Repository #908](https://archive.ics.uci.edu/dataset/908/realwaste)  
Size: ~650 MB zip → ~660 MB extracted  
License: CC BY 4.0

**Skip this cell if the dataset folder already exists.**

In [ ]:
import urllib.request
import zipfile
from tqdm import tqdm

DATASET_URL  = 'https://archive.ics.uci.edu/static/public/908/realwaste.zip'
ZIP_PATH     = DATA_DIR / 'realwaste.zip'
DATASET_DIR  = DATA_DIR / 'RealWaste'

class ProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

if DATASET_DIR.exists():
    print(f'✅ Dataset already exists at {DATASET_DIR} — skipping download.')
else:
    print('Downloading RealWaste dataset (~650 MB)...')
    with ProgressBar(unit='B', unit_scale=True, miniters=1, desc='RealWaste') as pb:
        urllib.request.urlretrieve(DATASET_URL, filename=ZIP_PATH, reporthook=pb.update_to)
    
    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_DIR)
    
    ZIP_PATH.unlink()  # delete zip to save space
    print(f'✅ Dataset extracted to {DATASET_DIR}')

## 0.5 — Verify Dataset Structure & Class Counts

In [ ]:
import os
from pathlib import Path
import pandas as pd

# Find the actual dataset folder (might be nested inside a subdirectory)
# Walk until we find a folder containing class subfolders with images
EXPECTED_CLASSES = {
    'Cardboard', 'Food Organics', 'Glass', 'Metal',
    'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation'
}

def find_dataset_root(base: Path) -> Path:
    """Recursively find the folder that contains the 9 class subdirectories."""
    for root, dirs, files in os.walk(base):
        if EXPECTED_CLASSES.issubset(set(dirs)):
            return Path(root)
    raise FileNotFoundError(f'Could not find dataset class folders under {base}')

DATASET_ROOT = find_dataset_root(DATA_DIR)
print(f'Dataset root found at: {DATASET_ROOT}\n')

# Count images per class
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}
records = []
for cls_dir in sorted(DATASET_ROOT.iterdir()):
    if cls_dir.is_dir():
        images = [f for f in cls_dir.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS]
        records.append({'class': cls_dir.name, 'count': len(images)})

df_counts = pd.DataFrame(records).sort_values('count', ascending=False).reset_index(drop=True)
df_counts['percentage'] = (df_counts['count'] / df_counts['count'].sum() * 100).round(2)

print(df_counts.to_string(index=False))
print(f'\nTotal images : {df_counts["count"].sum()}')
print(f'Majority class baseline accuracy: {df_counts["percentage"].max():.2f}%')

## 0.6 — Duplicate Check

Compute MD5 hash for every image and detect exact duplicates before splitting.

In [ ]:
import hashlib
from collections import defaultdict

def md5_hash(filepath: Path) -> str:
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

print('Computing MD5 hashes for all images (may take ~1-2 min)...')
hash_to_files = defaultdict(list)

all_image_paths = sorted([
    p for p in DATASET_ROOT.rglob('*')
    if p.suffix.lower() in IMAGE_EXTENSIONS
])

for img_path in tqdm(all_image_paths, desc='Hashing'):
    h = md5_hash(img_path)
    hash_to_files[h].append(img_path)

duplicates = {h: paths for h, paths in hash_to_files.items() if len(paths) > 1}

if duplicates:
    total_dups = sum(len(v) - 1 for v in duplicates.values())
    print(f'⚠️  Found {len(duplicates)} duplicate groups ({total_dups} redundant images):')
    for h, paths in list(duplicates.items())[:5]:  # show first 5
        print(f'  Hash {h[:8]}...: {[str(p.name) for p in paths]}')
else:
    print('✅ No duplicate images found.')

# Build list of unique images (keep one from each duplicate group)
unique_images = [paths[0] for paths in hash_to_files.values()]
print(f'\nUnique images to use for splitting: {len(unique_images)}')

## 1.1 — Create Shared Stratified Split

- **70% train / 15% val / 15% test**
- **Fixed seed: 42** — all team members must use this
- Saved as CSV files — every notebook reads these same files

This ensures fair comparison: every model sees **exactly the same** training, validation, and test images.

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

# Build master dataframe of all unique images
df_all = pd.DataFrame([
    {'filepath': str(p), 'label': p.parent.name}
    for p in unique_images
])

# Encode labels as integers
classes = sorted(df_all['label'].unique())
label2idx = {c: i for i, c in enumerate(classes)}
idx2label = {i: c for c, i in label2idx.items()}
df_all['label_idx'] = df_all['label'].map(label2idx)

print(f'Total unique images : {len(df_all)}')
print(f'Classes ({len(classes)}): {classes}\n')

# Stratified split: 70% train, 15% val, 15% test
df_train, df_temp = train_test_split(
    df_all, test_size=0.30, stratify=df_all['label_idx'], random_state=SEED
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.50, stratify=df_temp['label_idx'], random_state=SEED
)

print(f'Train : {len(df_train):>5} images')
print(f'Val   : {len(df_val):>5} images')
print(f'Test  : {len(df_test):>5} images')

## 1.2 — Verify Split is Stratified

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

# Per-class distribution across splits
split_summary = []
for split_name, df_split in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    for cls in classes:
        count = (df_split['label'] == cls).sum()
        split_summary.append({'split': split_name, 'class': cls, 'count': count})

df_summary = pd.DataFrame(split_summary)
pivot = df_summary.pivot(index='class', columns='split', values='count')
pivot['Total'] = pivot.sum(axis=1)
print(pivot.to_string())

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(classes))
width = 0.25
ax.bar([i - width for i in x], pivot['Train'], width=width, label='Train', color='steelblue')
ax.bar(x,                       pivot['Val'],   width=width, label='Val',   color='orange')
ax.bar([i + width for i in x], pivot['Test'],  width=width, label='Test',  color='green')
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=30, ha='right')
ax.set_ylabel('Image Count')
ax.set_title('Stratified Split — Images per Class')
ax.legend()
plt.tight_layout()

plot_path = PLOTS_DIR / 'split_distribution.png'
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'\nPlot saved to {plot_path}')

## 1.3 — Save Split CSVs & Metadata

In [ ]:
import json

# Save split CSVs
df_train.to_csv(DATA_DIR / 'train.csv', index=False)
df_val.to_csv(DATA_DIR  / 'val.csv',   index=False)
df_test.to_csv(DATA_DIR / 'test.csv',  index=False)

# Save class mapping
meta = {
    'seed'       : SEED,
    'split'      : '70/15/15',
    'n_train'    : len(df_train),
    'n_val'      : len(df_val),
    'n_test'     : len(df_test),
    'n_total'    : len(df_all),
    'n_classes'  : len(classes),
    'classes'    : classes,
    'label2idx'  : label2idx,
    'dataset_root': str(DATASET_ROOT)
}

with open(DATA_DIR / 'dataset_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('✅ Saved:')
print(f'   {DATA_DIR}/train.csv        ({len(df_train)} rows)')
print(f'   {DATA_DIR}/val.csv          ({len(df_val)} rows)')
print(f'   {DATA_DIR}/test.csv         ({len(df_test)} rows)')
print(f'   {DATA_DIR}/dataset_meta.json')
print()
print('🎯 Phase 0+1 complete! You can now run the other notebooks.')

## ✅ Summary — What Was Created

| File | Description |
|---|---|
| `data/train.csv` | Training set filepaths + labels (70%) |
| `data/val.csv` | Validation set filepaths + labels (15%) |
| `data/test.csv` | Test set filepaths + labels (15%) |
| `data/dataset_meta.json` | Class mapping, seed, split sizes |
| `results/plots/split_distribution.png` | Class distribution plot |

**All subsequent notebooks load these CSVs to guarantee identical train/val/test splits.**

Next: Open `phase2_classical_baselines.ipynb`